# Sistem Pendukung Keputusan (SPK) Distribusi Obat Darurat
## Metode: Fuzzy Mamdani + Multi-Attribute Decision Making (MADM)

Notebook ini memuat visualisasi dan kalkulasi logika fuzzy backend yang diambil dari aplikasi `app.py` untuk mempermudah proses debugging dan penjelasan mekanisme perhitungan.

In [9]:
# 1. IMPORT LIBRARY & PREPARASI DATA DUMMY
import pandas as pd
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt

print("Library berhasil dimuat!")

# Membuat data dummy pengganti file upload agar code langsung bisa running
data_dummy = {
    'ID_Alternatif': ['A01', 'A02', 'A03', 'A04', 'A05'],
    'Moda_Transportasi': ['Ambulans Motor', 'Ambulans Mobil', 'Helikopter Medis', 'Drone Logistik', 'Mobil Box Farmasi'],
    'C1_Waktu': [30, 65, 15, 45, 95],          # Menit (0-150)
    'C2_Biaya': [300, 700, 1900, 1100, 600],    # Ribu Rupiah (0-2000)
    'C3_Cuaca': [4, 3, 2, 5, 4],                # Skala 1-5
    'C4_Kapasitas': [2, 4, 5, 1, 5],            # Skala 1-5
    'C5_Medan': [4, 3, 5, 5, 2]                 # Skala 1-5
}

df_raw = pd.DataFrame(data_dummy)
df_raw

Library berhasil dimuat!


,ID_Alternatif,Moda_Transportasi,C1_Waktu,C2_Biaya,C3_Cuaca,C4_Kapasitas,C5_Medan
0,A01,Ambulans Motor,30,300,4,2,4
1,A02,Ambulans Mobil,65,700,3,4,3
2,A03,Helikopter Medis,15,1900,2,5,5
3,A04,Drone Logistik,45,1100,5,1,5
4,A05,Mobil Box Farmasi,95,600,4,5,2


# 2. INISIALISASI VARIABEL FUZZY (ANTECEDENT & CONSEQUENT)
Di sini kita mendefinisikan semesta pembicaraan (*universe*) untuk 5 kriteria input ($C_1$ sampai $C_5$) dan 1 output (Skor Kelayakan).

In [10]:
# Input (Antecedent)
waktu = ctrl.Antecedent(np.arange(0, 151, 1), 'C1')
biaya = ctrl.Antecedent(np.arange(0, 2001, 1), 'C2')
cuaca = ctrl.Antecedent(np.arange(1, 6, 1), 'C3')
kapasitas = ctrl.Antecedent(np.arange(1, 6, 1), 'C4')
medan = ctrl.Antecedent(np.arange(1, 6, 1), 'C5')

# Output (Consequent)
skor = ctrl.Consequent(np.arange(0, 101, 1), 'Membership')

print("Variabel Fuzzy berhasil diinisialisasi!")

Variabel Fuzzy berhasil diinisialisasi!


# 3. DEFINISI MEMBERSHIP FUNCTION (FUNGSI KEANGGOTAAN)
Menentukan kurva fungsi keanggotaan menggunakan tipe Segitiga (*trimf*) dan Trapesium (*trapmf*) sesuai parameter di `app.py`.

In [11]:
# Menentukan fungsi keanggotaan Waktu (C1)
waktu['cepat'] = fuzz.trimf(waktu.universe, [0, 0, 60])
waktu['sedang'] = fuzz.trapmf(waktu.universe, [40, 60, 90, 110])
waktu['lambat'] = fuzz.trimf(waktu.universe, [90, 150, 150])

# Menentukan fungsi keanggotaan Biaya (C2)
biaya['murah'] = fuzz.trimf(biaya.universe, [0, 0, 800])
biaya['standar'] = fuzz.trimf(biaya.universe, [500, 1000, 1500])
biaya['mahal'] = fuzz.trimf(biaya.universe, [1200, 2000, 2000])

# Menentukan fungsi keanggotaan Cuaca, Kapasitas, Medan (C3, C4, C5) secara looping
for k in [cuaca, kapasitas, medan]:
    k['buruk'] = fuzz.trimf(k.universe, [1, 1, 3])
    k['cukup'] = fuzz.trimf(k.universe, [2, 3, 4])
    k['bagus'] = fuzz.trimf(k.universe, [3, 5, 5])

# Menentukan fungsi keanggotaan Output Skor
skor['sedikit'] = fuzz.trimf(skor.universe, [0, 0, 45])
skor['sedang'] = fuzz.trimf(skor.universe, [35, 55, 75])
skor['banyak'] = fuzz.trimf(skor.universe, [60, 100, 100])

print("Membership function berhasil dipetakan!")

Membership function berhasil dipetakan!


# 4. PEMBENTUKAN ATURAN FUZZY (RULES BASE) & KONTROL SISTEM
Menerapkan 10 aturan dasar penalaran logika fuzzy Mamdani untuk menentukan tingkat kelayakan moda distribusi.

In [12]:
rules = [
    ctrl.Rule(waktu['cepat'] & biaya['murah'], skor['banyak']),
    ctrl.Rule(waktu['cepat'] & cuaca['bagus'], skor['banyak']),
    ctrl.Rule(waktu['cepat'] & medan['bagus'], skor['banyak']),
    ctrl.Rule(kapasitas['bagus'] & cuaca['bagus'], skor['banyak']),
    ctrl.Rule(biaya['mahal'] | waktu['lambat'], skor['sedikit']),
    ctrl.Rule(medan['buruk'] & cuaca['buruk'], skor['sedikit']),
    ctrl.Rule(kapasitas['buruk'] & biaya['mahal'], skor['sedikit']),
    ctrl.Rule(waktu['sedang'] & biaya['standar'], skor['sedang']),
    ctrl.Rule(cuaca['cukup'] & medan['cukup'], skor['sedang']),
    ctrl.Rule(kapasitas['cukup'], skor['sedang'])
]

# Satukan rule ke dalam sistem kontrol
fuzzy_system = ctrl.ControlSystem(rules)
print(f"Sistem kontrol fuzzy berhasil dibuat dengan {len(rules)} aturan.")

Sistem kontrol fuzzy berhasil dibuat dengan 10 aturan.


# 5. INPUT PARAMETER TARGET DAN CONFIG BOBOT PRIORITAS
Tahap simulasi interaksi user (menggantikan slider di Streamlit). Kamu bisa mengubah nilai variabel di bawah ini untuk mensimulasikan kebutuhan target operasional yang berbeda.

In [13]:
# Simulasi Nilai Ideal / Target Pengambil Keputusan
val_c1 = 45   # Target Waktu ideal (menit)
val_c2 = 600  # Target Biaya ideal (rupiah)
val_c3 = 4    # Target Kondisi Cuaca (1-5)
val_c4 = 3    # Target Kapasitas Kendaraan (1-5)
val_c5 = 4    # Target Kondisi Medan (1-5)

# Simulasi Penentuan Prioritas Kriteria (True = Bobot 2, False = Bobot 1)
prioritas_c1 = True   # Contoh: Memprioritaskan Waktu
prioritas_c2 = False
prioritas_c3 = False
prioritas_c4 = False
prioritas_c5 = False

# Kalkulasi Nilai Bobot Real
bobot_c1 = 2 if prioritas_c1 else 1
bobot_c2 = 2 if prioritas_c2 else 1
bobot_c3 = 2 if prioritas_c3 else 1
bobot_c4 = 2 if prioritas_c4 else 1
bobot_c5 = 2 if prioritas_c5 else 1

print("Parameter target pengambilan keputusan berhasil dikonfigurasi!")

Parameter target pengambilan keputusan berhasil dikonfigurasi!


# 6. PROSES INFERENSI FUZZY (ITERASI DATASET)
Menghitung `Skor_Fuzzy` untuk setiap baris data yang ada pada dataset secara sekuensial.

In [14]:
hasil_fuzzy = []

for i, row in df_raw.iterrows():
    sim = ctrl.ControlSystemSimulation(fuzzy_system)
    
    # Input data alternatif ke mesin fuzzy
    sim.input['C1'] = row['C1_Waktu']
    sim.input['C2'] = row['C2_Biaya']
    sim.input['C3'] = row['C3_Cuaca']
    sim.input['C4'] = row['C4_Kapasitas']
    sim.input['C5'] = row['C5_Medan']
    
    try:
        sim.compute()
        score = sim.output['Membership']
    except Exception as e:
        score = 0  # Fallback jika terjadi error diluar domain/rule tidak terpenuhi
        
    hasil_fuzzy.append(score)

df_hasil = df_raw.copy()
df_hasil['Skor_Fuzzy'] = hasil_fuzzy
df_hasil[['ID_Alternatif', 'Moda_Transportasi', 'Skor_Fuzzy']]

,ID_Alternatif,Moda_Transportasi,Skor_Fuzzy
0,A01,Ambulans Motor,84.444444
1,A02,Ambulans Mobil,55.000000
2,A03,Helikopter Medis,47.662966
3,A04,Drone Logistik,68.733333
4,A05,Mobil Box Farmasi,68.121720


# 7. PERHITUNGAN PERSENTASE KECOCOKAN (JARAK EUCLIDEAN BERBOBOT)
Menghitung seberapa dekat spesifikasi alternatif moda transportasi dengan kriteria ideal yang diinginkan oleh user menggunakan rumus Jarak Euclidean yang dikalikan dengan bobot prioritas.

In [15]:
# Rumus Euclidean Distance Berbobot
jarak = np.sqrt(
    bobot_c1 * ((df_hasil['C1_Waktu'] - val_c1) ** 2) +
    bobot_c2 * ((df_hasil['C2_Biaya'] - val_c2) ** 2) +
    bobot_c3 * ((df_hasil['C3_Cuaca'] - val_c3) ** 2) +
    bobot_c4 * ((df_hasil['C4_Kapasitas'] - val_c4) ** 2) +
    bobot_c5 * ((df_hasil['C5_Medan'] - val_c5) ** 2)
)

max_jarak = jarak.max()

# Normalisasi jarak menjadi persentase kecocokan (0-100%)
df_hasil['Persentase_Kecocokan'] = np.clip(
    100 * (1 - (jarak / max_jarak)), 0, 100
)

df_hasil[['ID_Alternatif', 'Moda_Transportasi', 'Persentase_Kecocokan']]

,ID_Alternatif,Moda_Transportasi,Persentase_Kecocokan
0,A01,Ambulans Motor,76.877701
1,A02,Ambulans Mobil,92.009085
2,A03,Helikopter Medis,0.000000
3,A04,Drone Logistik,61.558569
4,A05,Mobil Box Farmasi,94.559279


# 8. PERHITUNGAN SKOR AKHIR & PERANKINGAN ALTERNATIF
Menggabungkan nilai `Persentase_Kecocokan` (bobot 70%) dengan `Skor_Fuzzy` (bobot 30%) untuk mendapatkan nilai rekomendasi mutlak, lalu mengurutkannya untuk proses ranking.

In [16]:
# Pembobotan kombinasi MADM & Fuzzy
df_hasil['Skor_Akhir'] = (0.7 * df_hasil['Persentase_Kecocokan']) + (0.3 * df_hasil['Skor_Fuzzy'])

# Pengurutan data berdasarkan skor akhir tertinggi
df_hasil = df_hasil.sort_values(by='Skor_Akhir', ascending=False).reset_index(drop=True)
df_hasil.index += 1
df_hasil['Ranking'] = df_hasil.index

# Fungsi Pelabelan Bintang & Status Rekomendasi
def stars(score):
    if score >= 90: return "⭐⭐⭐⭐⭐"
    elif score >= 80: return "⭐⭐⭐⭐"
    elif score >= 70: return "⭐⭐⭐"
    elif score >= 60: return "⭐⭐"
    else: return "⭐"

def status(score):
    if score >= 90: return "Sangat Direkomendasikan"
    elif score >= 80: return "Direkomendasikan"
    elif score >= 70: return "Cukup Direkomendasikan"
    elif score >= 60: return "Kurang Direkomendasikan"
    else: return "Tidak Direkomendasikan"

df_hasil['Bintang'] = df_hasil['Skor_Akhir'].apply(stars)
df_hasil['Keterangan'] = df_hasil['Skor_Akhir'].apply(status)

# Menampilkan Hasil Akhir Rekomendasi SPK
Hasil_Akhir = df_hasil[[
    'Ranking', 'ID_Alternatif', 'Moda_Transportasi', 
    'Persentase_Kecocokan', 'Skor_Fuzzy', 'Skor_Akhir', 
    'Bintang', 'Keterangan'
]]
Hasil_Akhir

,Ranking,ID_Alternatif,Moda_Transportasi,Persentase_Kecocokan,Skor_Fuzzy,Skor_Akhir,Bintang,Keterangan
1,1,A05,Mobil Box Farmasi,94.559279,68.121720,86.628011,⭐⭐⭐⭐,Direkomendasikan
2,2,A02,Ambulans Mobil,92.009085,55.000000,80.906359,⭐⭐⭐⭐,Direkomendasikan
3,3,A01,Ambulans Motor,76.877701,84.444444,79.147724,⭐⭐⭐,Cukup Direkomendasikan
4,4,A04,Drone Logistik,61.558569,68.733333,63.710998,⭐⭐,Kurang Direkomendasikan
5,5,A03,Helikopter Medis,0.000000,47.662966,14.298890,⭐,Tidak Direkomendasikan
